In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

In [2]:
SEP      = ";"
ENC      = "latin-1"
SERVICO  = "141"
CLASS    = "002"
COMP     = "202603"

In [3]:
print("=" * 55)
print("  ANÁLISE: Profissionais em Vigilância Sanitária")
print(f"  Competência: {COMP[:4]}/{COMP[4:]}  |  Serviço 141 / 002")
print("=" * 55)

  ANÁLISE: Profissionais em Vigilância Sanitária
  Competência: 2026/03  |  Serviço 141 / 002


In [4]:
# ── 1. FILTRAR ESTABELECIMENTOS COM SERVIÇO 141/002 ──────────────────────────
print("\n[1/5] Filtrando estabelecimentos (Serviço 141 / Classificação 002)...")

serv = pd.read_csv(
    f"rlEstabServClass{COMP}.csv", sep=SEP, encoding=ENC, dtype=str,
    usecols=["CO_UNIDADE", "CO_SERVICO", "CO_CLASSIFICACAO"]
)
serv = serv.apply(lambda col: col.str.strip())

estab_visa = serv[
    (serv["CO_SERVICO"] == SERVICO) & (serv["CO_CLASSIFICACAO"] == CLASS)
]["CO_UNIDADE"].unique()

print(f"    Estabelecimentos encontrados: {len(estab_visa):,}")


[1/5] Filtrando estabelecimentos (Serviço 141 / Classificação 002)...
    Estabelecimentos encontrados: 13,991


In [5]:
# ── 2. PEGAR MUNICÍPIO E ESTADO DE CADA ESTABELECIMENTO ──────────────────────
print("\n[2/5] Carregando dados dos estabelecimentos...")

estab = pd.read_csv(
    f"tbEstabelecimento{COMP}.csv", sep=SEP, encoding=ENC, dtype=str,
    usecols=["CO_UNIDADE", "CO_MUNICIPIO_GESTOR", "CO_ESTADO_GESTOR"]
)
estab = estab[estab["CO_UNIDADE"].isin(estab_visa)].copy()
estab = estab.apply(lambda col: col.str.strip())
# zfill para garantir 6 dígitos no código de município
estab["CO_MUNICIPIO_GESTOR"] = estab["CO_MUNICIPIO_GESTOR"].str.zfill(6)
estab["CO_ESTADO_GESTOR"]    = estab["CO_ESTADO_GESTOR"].str.zfill(2)


[2/5] Carregando dados dos estabelecimentos...


In [6]:
# ── 3. PROFISSIONAIS VINCULADOS (CARGA HORÁRIA) ───────────────────────────────
print("\n[3/5] Carregando carga horária dos profissionais...")

carga = pd.read_csv(
    f"tbCargaHorariaSus{COMP}.csv", sep=SEP, encoding=ENC, dtype=str,
    usecols=["CO_UNIDADE", "CO_PROFISSIONAL_SUS"]
)
carga = carga.apply(lambda col: col.str.strip())

# Manter apenas profissionais dos estabelecimentos VISA
carga_visa = carga[carga["CO_UNIDADE"].isin(estab_visa)].copy()
print(f"    Vínculos encontrados nos estab. VISA: {len(carga_visa):,}")


[3/5] Carregando carga horária dos profissionais...
    Vínculos encontrados nos estab. VISA: 443,821


In [7]:
# ── 4. GARANTIR UNICIDADE DE PROFISSIONAIS (sem duplicatas) ──────────────────
print("\n[4/5] Removendo duplicatas de profissionais...")

# Um profissional pode atuar em mais de um estabelecimento VISA.
# Para contar INDIVÍDUOS únicos por local, a lógica correta é:
#   - Por município: conta CO_PROFISSIONAL_SUS distintos NO município
#   - Por estado: conta CO_PROFISSIONAL_SUS distintos NO estado
# Assim evitamos contar a mesma pessoa duas vezes dentro do mesmo nível geográfico.

# Unir carga com município/estado
carga_visa = carga_visa.merge(
    estab[["CO_UNIDADE", "CO_MUNICIPIO_GESTOR", "CO_ESTADO_GESTOR"]],
    on="CO_UNIDADE", how="left"
)

# Carregar nomes de municípios
mun = pd.read_csv(
    f"tbMunicipio{COMP}.csv", sep=SEP, encoding=ENC, dtype=str,
    usecols=["CO_MUNICIPIO", "NO_MUNICIPIO", "CO_SIGLA_ESTADO"]
)
mun = mun.apply(lambda col: col.str.strip())
mun["CO_MUNICIPIO"] = mun["CO_MUNICIPIO"].str.zfill(6)

# Carregar nomes de estados
estado = pd.read_csv(
    f"tbEstado{COMP}.csv", sep=SEP, encoding=ENC, dtype=str,
    usecols=["CO_UF", "CO_SIGLA", "NO_DESCRICAO"]
)
estado = estado.apply(lambda col: col.str.strip())
estado["CO_UF"] = estado["CO_UF"].str.zfill(2)

# Unir nomes
carga_visa = carga_visa.merge(
    mun.rename(columns={"CO_MUNICIPIO": "CO_MUNICIPIO_GESTOR"}),
    on="CO_MUNICIPIO_GESTOR", how="left"
)
carga_visa = carga_visa.merge(
    estado.rename(columns={"CO_UF": "CO_ESTADO_GESTOR",
                            "CO_SIGLA": "SIGLA_UF",
                            "NO_DESCRICAO": "NO_ESTADO"}),
    on="CO_ESTADO_GESTOR", how="left"
)

# Total Brasil: profissionais únicos no país inteiro
total_brasil = carga_visa["CO_PROFISSIONAL_SUS"].nunique()
print(f"    Total de profissionais únicos (Brasil): {total_brasil:,}")


[4/5] Removendo duplicatas de profissionais...
    Total de profissionais únicos (Brasil): 421,076


In [8]:
# ── 5. AGREGAÇÕES SEM DUPLICATA ───────────────────────────────────────────────

# Por Estado: profissionais únicos dentro de cada estado
por_estado = (
    carga_visa.groupby(["SIGLA_UF", "NO_ESTADO"], dropna=False)["CO_PROFISSIONAL_SUS"]
    .nunique()
    .reset_index()
    .rename(columns={"SIGLA_UF": "UF", "NO_ESTADO": "Estado",
                     "CO_PROFISSIONAL_SUS": "Qtd_Profissionais"})
    .sort_values("Estado")
)

# Linha de total Brasil
total_row_estado = pd.DataFrame([{"UF": "BR", "Estado": "TOTAL BRASIL",
                                   "Qtd_Profissionais": total_brasil}])
por_estado = pd.concat([total_row_estado, por_estado], ignore_index=True)

# Por Município: profissionais únicos dentro de cada município
por_municipio = (
    carga_visa.groupby(
        ["CO_MUNICIPIO_GESTOR", "NO_MUNICIPIO", "SIGLA_UF", "NO_ESTADO"], dropna=False
    )["CO_PROFISSIONAL_SUS"]
    .nunique()
    .reset_index()
    .rename(columns={"CO_MUNICIPIO_GESTOR": "Cod_IBGE",
                     "NO_MUNICIPIO": "Município",
                     "SIGLA_UF": "UF",
                     "NO_ESTADO": "Estado",
                     "CO_PROFISSIONAL_SUS": "Qtd_Profissionais"})
    .sort_values(["Estado", "Município"])
)

total_row_mun = pd.DataFrame([{"Cod_IBGE": "", "Município": "TOTAL BRASIL",
                                "UF": "BR", "Estado": "",
                                "Qtd_Profissionais": total_brasil}])
por_municipio = pd.concat([total_row_mun, por_municipio], ignore_index=True)

print(f"\n    Estados com registros: {len(por_estado) - 1}")
print(f"    Municípios com registros: {len(por_municipio) - 1}")


    Estados com registros: 27
    Municípios com registros: 5561


In [9]:
# ── 6. EXPORTAR E FORMATAR EXCEL ──────────────────────────────────────────────
print("\n[5/5] Gerando relatórios Excel...")

def salvar_excel(df, nome_arquivo, titulo):
    df.to_excel(nome_arquivo, index=False, sheet_name="Dados")

    wb = load_workbook(nome_arquivo)
    ws = wb.active
    ws.title = "Dados"

    COR_HEADER  = "1F4E79"
    COR_TOTAL   = "FFF2CC"
    COR_PAR     = "D6E4F0"
    COR_IMPAR   = "FFFFFF"

    borda = Border(bottom=Side(style="thin", color="BFBFBF"))

    # Cabeçalho
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
        cell.fill      = PatternFill("solid", start_color=COR_HEADER)
        cell.alignment = Alignment(horizontal="center", vertical="center",
                                   wrap_text=True)
        cell.border    = borda
    ws.row_dimensions[1].height = 32

    # Linhas de dados
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        is_total = str(ws.cell(row=i, column=1).value) in ("BR", "")
        for j, cell in enumerate(row, start=1):
            if is_total:
                cell.fill = PatternFill("solid", start_color=COR_TOTAL)
                cell.font = Font(bold=True, name="Arial", size=10)
            else:
                cor = COR_PAR if i % 2 == 0 else COR_IMPAR
                cell.fill = PatternFill("solid", start_color=cor)
                cell.font = Font(name="Arial", size=10)
            cell.alignment = Alignment(vertical="center")
            cell.border    = borda
            # Números: alinhar à direita com separador de milhar
            if isinstance(cell.value, (int, float)):
                cell.alignment  = Alignment(horizontal="right", vertical="center")
                cell.number_format = "#,##0"

    # Ajustar largura das colunas
    for col in ws.columns:
        max_len = max(
            (len(str(c.value)) if c.value is not None else 0) for c in col
        )
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 45)

    # Fixar primeira linha
    ws.freeze_panes = "A2"

    # Adicionar aba de metadados
    ws_meta = wb.create_sheet("Sobre")
    meta = [
        ("Relatório",     titulo),
        ("Competência",   f"{COMP[:4]}/{COMP[4:]}"),
        ("Serviço CNES",  "141 - Vigilância em Saúde"),
        ("Classificação", "002 - Vigilância Sanitária"),
        ("Fonte",         "DATASUS / CNES"),
        ("Total Brasil",  total_brasil),
        ("Nota",          "Contagem de profissionais únicos (CO_PROFISSIONAL_SUS). "
                          "Um profissional que atua em mais de um estabelecimento "
                          "dentro do mesmo nível geográfico é contado apenas uma vez."),
    ]
    for r, (chave, valor) in enumerate(meta, start=1):
        ws_meta.cell(r, 1, chave).font  = Font(bold=True, name="Arial", size=10)
        ws_meta.cell(r, 2, valor).font  = Font(name="Arial", size=10)
        ws_meta.cell(r, 2).alignment    = Alignment(wrap_text=True)
    ws_meta.column_dimensions["A"].width = 18
    ws_meta.column_dimensions["B"].width = 70

    wb.save(nome_arquivo)
    print(f"    ✅ {nome_arquivo}")

salvar_excel(
    por_estado,
    "relatorio_visa_por_estado.xlsx",
    "Profissionais em Vigilância Sanitária por Estado"
)
salvar_excel(
    por_municipio,
    "relatorio_visa_por_municipio.xlsx",
    "Profissionais em Vigilância Sanitária por Município"
)

print(f"\n{'='*55}")
print(f"  CONCLUÍDO!")
print(f"  Total de profissionais únicos no Brasil: {total_brasil:,}")
print(f"  Arquivos gerados:")
print(f"    → relatorio_visa_por_estado.xlsx")
print(f"    → relatorio_visa_por_municipio.xlsx")
print(f"{'='*55}")


[5/5] Gerando relatórios Excel...
    ✅ relatorio_visa_por_estado.xlsx
    ✅ relatorio_visa_por_municipio.xlsx

  CONCLUÍDO!
  Total de profissionais únicos no Brasil: 421,076
  Arquivos gerados:
    → relatorio_visa_por_estado.xlsx
    → relatorio_visa_por_municipio.xlsx
